In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, List, Optional, Tuple
from pathlib import Path
import os
from glob import glob

import h5py
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import sys

# Add src to path for custom utilities
sys.path.insert(0, f'/home/obandohe/JIPSENSE/JIPSENSE')
from src.utils.seismic_io import load_traces_from_hdf5, print_hdf5_info
from src.utils.fk_spectrum_class import FKSpectrum
from src.utils.Preprocessing_SW import agc_gain

print("✓ All libraries imported successfully")

In [ ]:

def read_trace_file(semp_file: Path) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Read a SPECFEM2D seismic trace file (.semp format).
    
    Format: Two columns separated by whitespace
    - Column 1: Time (seconds)
    - Column 2: Pressure/Amplitude
    
    Args:
        semp_file: Path to .semp file
        
    Returns:
        Tuple of (time array, pressure array, dt)
    """
    try:
        data = np.loadtxt(semp_file)
        
        if data.ndim == 1:
            # Single sample (unlikely, but handle it)
            time = np.array([data[0]])
            pressure = np.array([data[1]])
        else:
            time = data[:, 0]
            pressure = data[:, 1]
        
        # Compute dt from time differences
        if len(time) > 1:
            dt = float(np.mean(np.diff(time)))
        else:
            dt = 1.0  # Default if only one sample
        
        return time, pressure, dt
    
    except Exception as e:
        print(f"Error reading {semp_file}: {e}")
        return None, None, None

In [ ]:

all_traces = []

for i in range(1,29):


    files = glob(f'/home/obandohe/JIPSENSE/JIPSENSE/src/simulation_standard/output_standard/OUTPUT_FILES_2026-08-24_11-08-09/AA.S{str(i).zfill(4)}.PRE.semp')
            

    time,trace,dt = read_trace_file(files[0])

    time = time[:70000]
    trace = trace[:70000]

    all_traces.append(trace)


all_traces =  np.array(all_traces).T



In [ ]:

plt.figure(figsize=(5,8))

all_traces_norm = all_traces/np.max(np.abs(all_traces),axis=0)

plt.imshow(all_traces_norm,aspect='auto',cmap='seismic')

In [5]:
# compute a fk spectrum

# apply AGC to the filtered data
shotgather = all_traces_norm  # Assuming pressure_data is the shot gather
agc_gate = 0.1 # seconds
fs = 1/dt  # Sampling frequency in Hz
dx = 1.0  # receiver spacing in meters
sg_agc = agc_gain(shotgather, fs, agc_gate,method='amplitude')

# Initialize FK spectrum object
fk = FKSpectrum(dx=dx,dt=dt)
FK, f, k, _, _ = fk.compute(sg_agc)

KeyboardInterrupt: 

In [ ]:
# plot fk spectrum
plt.figure(figsize=(4, 6))
plt.imshow(np.log10(np.abs(FK)), extent=[k[0], k[-1], f[0], f[-1]], aspect='auto', origin='lower', cmap='jet')
plt.colorbar(label='Amplitude')
plt.title('FK Spectrum of Seismic Data (AGC Applied)')
plt.xlabel('Wavenumber (1/m)')
plt.ylabel('Frequency (Hz)')
plt.ylim(0,500)  # Limit frequency axis to 1000 Hz for better visibility

# add a slopeline to indicate the expected dispersion curve
v = 1500  # m/s, example velocity
f_line = np.linspace(0, 1000, 300)
k_line = f_line / v

v_soil = 180 # m/s, example velocity for soil
f_line_soil = np.linspace(0, 1000, 300)
k_line_soil = f_line_soil / v_soil

plt.plot(-k_line,f_line, 'w--', label=f'v={v} m/s',color='white')
plt.plot(-k_line_soil, f_line_soil, 'w-.', label=f'v={v_soil} m/s',color='blue')
plt.legend()

plt.xlim(-0.5, 0.5)  # Limit wavenumber axis for better visibility

plt.show()

In [ ]:
dt